# 🚀 Notebook 08 — FastAPI Backend
Exposes the recommendation engine as a REST API.

**Endpoints:**
- `GET /recommend` — recommendations by position/style/budget
- `GET /similar/{player_name}` — KNN similar players
- `GET /hidden-gems` — undervalued talent
- `GET /compare` — side-by-side comparison
- `GET /players` — search/autocomplete
- `GET /health` — status check

In [ ]:
# ── Imports & paths ──────────────────────────────────────────────────
import pickle, json
import pandas as pd
import numpy as np
from pathlib import Path

ROOT      = Path('C:/Users/Lenovo/Scouti_Koora')
PROCESSED = ROOT / 'data' / 'processed'
MODELS    = PROCESSED  # all pkl files are in data/processed/

# Verify artifacts exist
required = [
    PROCESSED / 'player_snapshot.csv',
    PROCESSED / 'style_clusters.csv',
    PROCESSED / 'talent_scores.csv',
    PROCESSED / 'knn_similarity_model.pkl',
    PROCESSED / 'similarity_scaler.pkl',
    PROCESSED / 'similarity_features.pkl',
    PROCESSED / 'xgb_talent_model.pkl',
    PROCESSED / 'cluster_models.pkl',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    print('❌ Missing artifacts:')
    for m in missing: print(f'   {m}')
else:
    print('✅ All artifacts found')
    print(f'   PROCESSED: {PROCESSED}')

In [ ]:
# ── Load all artifacts ───────────────────────────────────────────────
snapshot  = pd.read_csv(PROCESSED / 'player_snapshot.csv', low_memory=False)
clusters  = pd.read_csv(PROCESSED / 'style_clusters.csv',  low_memory=False)
talent    = pd.read_csv(PROCESSED / 'talent_scores.csv',   low_memory=False)

with open(PROCESSED / 'knn_similarity_model.pkl', 'rb') as f:
    knn_model = pickle.load(f)
with open(PROCESSED / 'similarity_scaler.pkl', 'rb') as f:
    knn_scaler = pickle.load(f)
with open(PROCESSED / 'similarity_features.pkl', 'rb') as f:
    knn_features = pickle.load(f)
with open(PROCESSED / 'xgb_talent_model.pkl', 'rb') as f:
    xgb_model = pickle.load(f)
with open(PROCESSED / 'cluster_models.pkl', 'rb') as f:
    cluster_models = pickle.load(f)

# Merge snapshot with clusters and talent
df = snapshot.copy()
if 'style_label' not in df.columns and 'style_label' in clusters.columns:
    merge_cols = [c for c in ['Player','season'] if c in clusters.columns]
    df = df.merge(
        clusters[merge_cols + ['style_label','cluster_id','position_group']].drop_duplicates(),
        on=merge_cols, how='left'
    )
if 'predicted_talent' not in df.columns and 'predicted_talent' in talent.columns:
    df = df.merge(
        talent[['Player','season','predicted_talent','talent_label']].drop_duplicates(),
        on=['Player','season'], how='left'
    )

print(f'✅ Snapshot: {len(df):,} rows × {df.shape[1]} cols')
print(f'   Players: {df["Player"].nunique():,}')
if 'style_label' in df.columns:
    print(f'   Styles:  {sorted(df["style_label"].dropna().unique().tolist())}')
print(f'   Talent:  {df["predicted_talent"].notna().sum():,} scored')

In [ ]:
# ── Build player index for KNN ───────────────────────────────────────
player_index   = pd.read_csv(PROCESSED / 'player_index.csv')
feature_matrix = pd.read_csv(PROCESSED / 'feature_matrix_scaled.csv')

# Drop non-numeric columns (Player name may be stored in the matrix)
feature_matrix_num = feature_matrix.select_dtypes(include=[np.number])
X_scaled = feature_matrix_num.values

# Latest season per player for API queries
latest = (
    df.sort_values('season', ascending=False)
    .drop_duplicates('Player', keep='first')
    .reset_index(drop=True)
)

print(f'✅ Player index:    {len(player_index):,} entries')
print(f'✅ Feature matrix:  {X_scaled.shape}')
print(f'✅ Latest snapshot: {len(latest):,} players')
print()
# Show exact name format for well-known players
for name in ['Haaland','Mbappe','Lewandowski','Kane','Salah']:
    found = player_index[player_index['Player'].str.contains(name, case=False, na=False)]['Player'].tolist()
    print(f'  {name} → {found[:2]}')

In [ ]:
# ── Core engine functions ─────────────────────────────────────────────
import unicodedata, re

def norm(name):
    if pd.isna(name): return ''
    n = unicodedata.normalize('NFD', str(name))
    n = ''.join(c for c in n if unicodedata.category(c) != 'Mn')
    return n.lower().strip()

def find_player(name, data=None):
    """Find player row by name (exact then partial match)."""
    if data is None: data = latest
    n = norm(name)
    exact = data[data['Player'].apply(norm) == n]
    if not exact.empty: return exact.iloc[0]
    partial = data[data['Player'].apply(norm).str.contains(n, na=False)]
    if not partial.empty: return partial.iloc[0]
    return None

def find_player_idx(name):
    """Find player row index in player_index (exact then partial)."""
    n = norm(name)
    exact = player_index[player_index['Player'].apply(norm) == n]
    if not exact.empty: return exact.index[0], exact.iloc[0]
    partial = player_index[player_index['Player'].apply(norm).str.contains(n, na=False)]
    if not partial.empty: return partial.index[0], partial.iloc[0]
    return None, None

def recommend(position=None, style=None, max_budget_eur=None,
              max_age=None, min_age=None, reference_player=None,
              leagues=None, min_minutes=500, top_n=10):
    pool = latest.copy()
    pos_map = {'forward':'pos_FW','midfielder':'pos_MF',
               'defender':'pos_DF','goalkeeper':'pos_GK'}
    if position:
        col = pos_map.get(position.lower())
        if col and col in pool.columns: pool = pool[pool[col] == 1]
    if min_minutes:
        pool = pool[pd.to_numeric(pool.get('minutes_played', 0), errors='coerce').fillna(0) >= min_minutes]
    if max_age:
        pool = pool[pd.to_numeric(pool['Age'], errors='coerce').fillna(99) <= max_age]
    if min_age:
        pool = pool[pd.to_numeric(pool['Age'], errors='coerce').fillna(0) >= min_age]
    if max_budget_eur:
        mv = pd.to_numeric(pool['market_value_eur'], errors='coerce')
        pool = pool[mv.isna() | (mv <= max_budget_eur)]
    if leagues:
        if isinstance(leagues, str): leagues = [leagues]
        pool = pool[pool['league'].isin(leagues)]
    if style and 'style_label' in pool.columns:
        pool = pool[pool['style_label'] == style]
    if pool.empty: return pd.DataFrame()
    score_col = 'predicted_talent' if 'predicted_talent' in pool.columns else 'overall_score'
    pool = pool.sort_values(score_col, ascending=False)
    out_cols = ['Player','primary_pos','Squad','league','season','Age',
                'style_label','predicted_talent','talent_label',
                'market_value_eur','goals_p90','assists_p90',
                'tackles_won_p90','availability_pct']
    out_cols = [c for c in out_cols if c in pool.columns]
    return pool[out_cols].head(top_n).reset_index(drop=True)


def find_similar(player_name, top_n=10, same_position=True):
    """KNN similarity search with robust name matching."""
    ridx, ref_row = find_player_idx(player_name)
    if ridx is None:
        return {'error': f'Player not found: {player_name}. '
                f'Try: {player_index["Player"].sample(3).tolist()}'}
    # Pass DataFrame with feature names to avoid sklearn warning
    query_vec = pd.DataFrame(X_scaled[ridx:ridx+1], columns=knn_features)
    dists, idxs = knn_model.kneighbors(
        query_vec,
        n_neighbors=min(top_n + 30, len(X_scaled))
    )
    ref_pos = ref_row.get('primary_pos', '')
    results = []
    for dist, idx in zip(dists[0], idxs[0]):
        if idx == ridx: continue
        row = player_index.iloc[idx]
        if same_position and row.get('primary_pos') != ref_pos: continue
        results.append({
            'Player':         row['Player'],
            'similarity_pct': round((1 - dist) * 100, 1),
            'primary_pos':    row.get('primary_pos', ''),
            'Squad':          row.get('Squad', ''),
            'league':         row.get('league', ''),
            'season':         row.get('season', ''),
        })
        if len(results) >= top_n: break
    return results


def find_hidden_gems(position=None, max_value_eur=20_000_000,
                     min_talent_score=60, max_age=26, top_n=20):
    pool = latest.copy()
    score_col = 'predicted_talent' if 'predicted_talent' in pool.columns else 'overall_score'
    pos_map = {'forward':'pos_FW','midfielder':'pos_MF',
               'defender':'pos_DF','goalkeeper':'pos_GK'}
    if position:
        col = pos_map.get(position.lower())
        if col and col in pool.columns: pool = pool[pool[col] == 1]
    mv  = pd.to_numeric(pool['market_value_eur'], errors='coerce')
    ts  = pd.to_numeric(pool[score_col], errors='coerce')
    age = pd.to_numeric(pool['Age'], errors='coerce')
    pool = pool[(mv <= max_value_eur) & (ts >= min_talent_score) & (age <= max_age)]
    pool = pool.sort_values(score_col, ascending=False)
    out_cols = ['Player','primary_pos','Squad','league','season','Age',
                'style_label', score_col, 'market_value_eur',
                'goals_p90','assists_p90','tackles_won_p90']
    out_cols = [c for c in out_cols if c in pool.columns]
    return pool[out_cols].head(top_n).reset_index(drop=True)


def compare_players(player_a, player_b):
    pa = find_player(player_a)
    pb = find_player(player_b)
    if pa is None: return {'error': f'Not found: {player_a}'}
    if pb is None: return {'error': f'Not found: {player_b}'}
    cols = ['Player','primary_pos','Squad','league','season','Age',
            'goals_p90','assists_p90','g_a_p90','shots_p90',
            'shot_accuracy_pct','tackles_won_p90','interceptions_p90',
            'def_actions_p90','availability_pct',
            'market_value_eur','predicted_talent','style_label']
    cols = [c for c in cols if c in latest.columns]
    return pd.DataFrame([pa[cols], pb[cols]]).reset_index(drop=True)


def search_players(query, top_n=10):
    q = norm(query)
    results = latest[latest['Player'].apply(norm).str.contains(q, na=False)]
    out_cols = ['Player','primary_pos','Squad','league','Age','market_value_eur']
    out_cols = [c for c in out_cols if c in results.columns]
    return results[out_cols].head(top_n).to_dict(orient='records')

print('✅ Engine functions defined')

In [ ]:
# ── Smoke tests ──────────────────────────────────────────────────────
print('TEST 1: Budget striker u26 <€20M')
r1 = recommend(position='forward', max_budget_eur=20_000_000, max_age=26, top_n=5)
if r1.empty:
    print('  No results')
else:
    cols = [c for c in ['Player','Age','market_value_eur','predicted_talent','style_label'] if c in r1.columns]
    print(r1[cols].to_string(index=False))

print('\nTEST 2: Similar players (auto-detect name)')
# Try known players from player_index
test_players = ['Haaland','Mbappe','Lewandowski','Kane']
for name in test_players:
    r2 = find_similar(name, top_n=3)
    if isinstance(r2, list) and r2:
        print(f'  {name} → similar:')
        for p in r2: print(f"    {p['Player']} ({p['similarity_pct']}%)")
        break
    elif isinstance(r2, dict):
        print(f'  {name}: {r2["error"]}')

print('\nTEST 3: Hidden gems midfielders u25')
r3 = find_hidden_gems(position='midfielder', max_value_eur=15_000_000,
                      min_talent_score=60, max_age=25, top_n=5)
if r3.empty:
    print('  No results — lowering threshold')
    r3 = find_hidden_gems(position='midfielder', max_value_eur=20_000_000,
                          min_talent_score=55, max_age=26, top_n=5)
cols = [c for c in ['Player','Age','market_value_eur','predicted_talent'] if c in r3.columns]
print(r3[cols].to_string(index=False))

print('\nTEST 4: Compare two players')
# Find two real player names from latest
sample = latest['Player'].dropna().sample(2, random_state=42).tolist()
print(f'  Comparing: {sample[0]} vs {sample[1]}')
r4 = compare_players(sample[0], sample[1])
if isinstance(r4, pd.DataFrame):
    cols = [c for c in ['Player','goals_p90','assists_p90','market_value_eur','predicted_talent'] if c in r4.columns]
    print(r4[cols].to_string(index=False))
else:
    print(f'  Error: {r4}')

print('\n✅ Smoke tests complete')

In [ ]:
# ── Write FastAPI app to src/api/main.py ─────────────────────────────
API_DIR = ROOT / 'src' / 'api'
API_DIR.mkdir(parents=True, exist_ok=True)

api_code = '''
"""
ScoutIQ FastAPI Backend
Run with: uvicorn src.api.main:app --reload --port 8000
Docs at:  http://localhost:8000/docs
"""
from fastapi import FastAPI, Query, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from typing import Optional, List
import pickle, pandas as pd, numpy as np
import unicodedata, re
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────
ROOT      = Path(__file__).resolve().parents[2]
PROCESSED = ROOT / "data" / "processed"
MODELS    = ROOT / "models"

# ── App ───────────────────────────────────────────────────────────────
app = FastAPI(
    title="ScoutIQ API",
    description="AI-powered football talent discovery engine",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ── Load artifacts on startup ─────────────────────────────────────────
@app.on_event("startup")
async def load_models():
    global snapshot, player_index, X_scaled
    global knn_model, knn_scaler, knn_features
    global xgb_model, cluster_models

    snapshot      = pd.read_csv(PROCESSED / "player_snapshot.csv", low_memory=False)
    clusters      = pd.read_csv(PROCESSED / "style_clusters.csv",  low_memory=False)
    talent        = pd.read_csv(PROCESSED / "talent_scores.csv",   low_memory=False)
    player_index  = pd.read_csv(PROCESSED / "player_index.csv")
    feat_matrix   = pd.read_csv(PROCESSED / "feature_matrix_scaled.csv")
    X_scaled      = feat_matrix.values

    with open(MODELS / "knn_similarity_model.pkl", "rb") as f: knn_model   = pickle.load(f)
    with open(MODELS / "similarity_scaler.pkl",   "rb") as f: knn_scaler  = pickle.load(f)
    with open(MODELS / "similarity_features.pkl", "rb") as f: knn_features = pickle.load(f)
    with open(MODELS / "xgb_talent_model.pkl",    "rb") as f: xgb_model   = pickle.load(f)
    with open(MODELS / "cluster_models.pkl",       "rb") as f: cluster_models = pickle.load(f)

    # Merge all into snapshot
    if "style_label" not in snapshot.columns and "style_label" in clusters.columns:
        snapshot = snapshot.merge(
            clusters[["Player","season","style_label","cluster_id","position_group"]].drop_duplicates(),
            on=["Player","season"], how="left"
        )
    if "predicted_talent" not in snapshot.columns and "predicted_talent" in talent.columns:
        snapshot = snapshot.merge(
            talent[["Player","season","predicted_talent","talent_label"]].drop_duplicates(),
            on=["Player","season"], how="left"
        )

    global latest
    latest = (
        snapshot.sort_values("season", ascending=False)
        .drop_duplicates("Player", keep="first")
        .reset_index(drop=True)
    )

# ── Helpers ───────────────────────────────────────────────────────────
def norm(name):
    if not name: return ""
    n = unicodedata.normalize("NFD", str(name))
    n = "".join(c for c in n if unicodedata.category(c) != "Mn")
    return n.lower().strip()

def safe_json(df):
    return df.where(pd.notna(df), None).to_dict(orient="records")

# ── Endpoints ─────────────────────────────────────────────────────────
@app.get("/health")
def health():
    return {"status": "ok", "players": len(latest), "style_labels": sorted(latest["style_label"].dropna().unique().tolist())}


@app.get("/players")
def search_players(q: str = Query(..., min_length=2), top_n: int = 10):
    """Autocomplete player search."""
    results = latest[latest["Player"].apply(norm).str.contains(norm(q), na=False)]
    out_cols = ["Player","primary_pos","Squad","league","Age","market_value_eur","predicted_talent"]
    out_cols = [c for c in out_cols if c in results.columns]
    return safe_json(results[out_cols].head(top_n))


@app.get("/recommend")
def recommend_players(
    position:        Optional[str]   = None,
    style:           Optional[str]   = None,
    max_budget_eur:  Optional[float] = None,
    max_age:         Optional[int]   = None,
    min_age:         Optional[int]   = None,
    leagues:         Optional[str]   = None,
    min_minutes:     int             = 500,
    reference_player:Optional[str]   = None,
    top_n:           int             = 10,
):
    """Main recommendation endpoint."""
    pool = latest.copy()
    pos_map = {"forward":"pos_FW","midfielder":"pos_MF","defender":"pos_DF","goalkeeper":"pos_GK"}

    if position:
        col = pos_map.get(position.lower())
        if col and col in pool.columns: pool = pool[pool[col] == 1]

    if min_minutes:
        pool = pool[pd.to_numeric(pool.get("minutes_played",0), errors="coerce").fillna(0) >= min_minutes]

    if max_age:
        pool = pool[pd.to_numeric(pool["Age"], errors="coerce").fillna(99) <= max_age]

    if min_age:
        pool = pool[pd.to_numeric(pool["Age"], errors="coerce").fillna(0) >= min_age]

    if max_budget_eur:
        mv = pd.to_numeric(pool["market_value_eur"], errors="coerce")
        pool = pool[mv.isna() | (mv <= max_budget_eur)]

    if leagues:
        lg_list = [l.strip() for l in leagues.split(",")]
        pool = pool[pool["league"].isin(lg_list)]

    if style and "style_label" in pool.columns:
        pool = pool[pool["style_label"] == style]

    if pool.empty:
        return []

    score_col = "predicted_talent" if "predicted_talent" in pool.columns else "overall_score"
    pool = pool.sort_values(score_col, ascending=False)

    out_cols = ["Player","primary_pos","Squad","league","season","Age",
                "style_label","predicted_talent","talent_label",
                "market_value_eur","goals_p90","assists_p90",
                "tackles_won_p90","availability_pct"]
    out_cols = [c for c in out_cols if c in pool.columns]
    return safe_json(pool[out_cols].head(top_n))


@app.get("/similar/{player_name}")
def similar_players(player_name: str, top_n: int = 10, same_position: bool = True):
    """KNN similarity search."""
    ref = player_index[player_index["Player"].apply(norm) == norm(player_name)]
    if ref.empty:
        # Try partial match
        ref = player_index[player_index["Player"].apply(norm).str.contains(norm(player_name), na=False)]
    if ref.empty:
        raise HTTPException(status_code=404, detail=f"Player not found: {player_name}")

    ridx = ref.index[0]
    dists, idxs = knn_model.kneighbors(X_scaled[ridx:ridx+1],
                                        n_neighbors=min(top_n+20, len(X_scaled)))
    ref_pos = player_index.iloc[ridx].get("primary_pos", "")
    results = []
    for dist, idx in zip(dists[0], idxs[0]):
        if idx == ridx: continue
        row = player_index.iloc[idx]
        if same_position and row.get("primary_pos") != ref_pos: continue
        results.append({
            "Player": row["Player"],
            "similarity_pct": round((1 - dist) * 100, 1),
            "primary_pos": row.get("primary_pos",""),
            "Squad": row.get("Squad",""),
            "league": row.get("league",""),
            "season": row.get("season",""),
        })
        if len(results) >= top_n: break
    return results


@app.get("/hidden-gems")
def hidden_gems(
    position:         Optional[str]   = None,
    max_value_eur:    float           = 20_000_000,
    min_talent_score: float           = 60,
    max_age:          int             = 26,
    top_n:            int             = 20,
):
    """High talent + low market value."""
    pool = latest.copy()
    score_col = "predicted_talent" if "predicted_talent" in pool.columns else "overall_score"
    pos_map = {"forward":"pos_FW","midfielder":"pos_MF","defender":"pos_DF","goalkeeper":"pos_GK"}

    if position:
        col = pos_map.get(position.lower())
        if col and col in pool.columns: pool = pool[pool[col] == 1]

    mv  = pd.to_numeric(pool["market_value_eur"], errors="coerce")
    ts  = pd.to_numeric(pool[score_col], errors="coerce")
    age = pd.to_numeric(pool["Age"], errors="coerce")
    pool = pool[(mv <= max_value_eur) & (ts >= min_talent_score) & (age <= max_age)]
    pool = pool.sort_values(score_col, ascending=False)

    out_cols = ["Player","primary_pos","Squad","league","season","Age",
                "style_label", score_col, "market_value_eur",
                "goals_p90","assists_p90","tackles_won_p90"]
    out_cols = [c for c in out_cols if c in pool.columns]
    return safe_json(pool[out_cols].head(top_n))


@app.get("/compare")
def compare(
    player_a: str = Query(...),
    player_b: str = Query(...),
):
    """Side-by-side player comparison."""
    def get_player(name):
        r = latest[latest["Player"].apply(norm) == norm(name)]
        if r.empty:
            r = latest[latest["Player"].apply(norm).str.contains(norm(name), na=False)]
        if r.empty: raise HTTPException(404, f"Not found: {name}")
        return r.iloc[0]

    pa, pb = get_player(player_a), get_player(player_b)
    cols = ["Player","primary_pos","Squad","league","Age",
            "goals_p90","assists_p90","g_a_p90","shots_p90",
            "shot_accuracy_pct","tackles_won_p90","interceptions_p90",
            "def_actions_p90","availability_pct",
            "market_value_eur","predicted_talent","style_label"]
    cols = [c for c in cols if c in latest.columns]
    result = pd.DataFrame([pa[cols], pb[cols]])
    return safe_json(result)


@app.get("/styles")
def get_styles():
    """List all available style labels."""
    if "style_label" not in latest.columns:
        return []
    return sorted(latest["style_label"].dropna().unique().tolist())
'''

with open(API_DIR / 'main.py', 'w', encoding='utf-8') as f:
    f.write(api_code.strip())

# Create __init__.py
(API_DIR / '__init__.py').touch()

print(f'✅ FastAPI app written to: {API_DIR / "main.py"}')
print(f'   Size: {(API_DIR / "main.py").stat().st_size:,} bytes')

In [ ]:
# ── Save recommendation engine as pickle for reuse ────────────────────
engine_bundle = {
    'snapshot':       snapshot,
    'latest':         latest,
    'player_index':   player_index,
    'X_scaled':       X_scaled,
    'knn_model':      knn_model,
    'knn_features':   knn_features,
    'xgb_model':      xgb_model,
    'cluster_models': cluster_models,
    'style_labels':   sorted(latest['style_label'].dropna().unique().tolist()) if 'style_label' in latest.columns else [],
}

with open(MODELS / 'recommendation_engine.pkl', 'wb') as f:
    pickle.dump(engine_bundle, f)

print(f'✅ Engine bundle saved: {MODELS / "recommendation_engine.pkl"}')
size_mb = (MODELS / 'recommendation_engine.pkl').stat().st_size / 1e6
print(f'   Size: {size_mb:.1f} MB')

In [ ]:
# ── Create requirements for API ───────────────────────────────────────
api_requirements = """fastapi>=0.110.0
uvicorn[standard]>=0.27.0
pandas>=2.0.0
numpy>=1.26.0
scikit-learn>=1.4.0
xgboost>=2.0.0
python-multipart>=0.0.9
"""

with open(ROOT / 'requirements_api.txt', 'w') as f:
    f.write(api_requirements)

print('✅ requirements_api.txt written')
print()
print('='*60)
print('  🚀 TO LAUNCH THE API:')
print('='*60)
print(f'  cd {ROOT}')
print('  pip install -r requirements_api.txt')
print('  uvicorn src.api.main:app --reload --port 8000')
print()
print('  📖 Swagger UI:  http://localhost:8000/docs')
print('  📊 ReDoc:       http://localhost:8000/redoc')
print()
print('  Endpoints:')
print('    GET /health')
print('    GET /players?q=mbappe')
print('    GET /recommend?position=forward&max_budget_eur=30000000&max_age=25')
print('    GET /similar/Haaland')
print('    GET /hidden-gems?position=midfielder&max_age=23')
print('    GET /compare?player_a=Mbappe&player_b=Haaland')
print('    GET /styles')

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────
summary = {
    'api': {
        'framework': 'FastAPI',
        'file': 'src/api/main.py',
        'endpoints': [
            'GET /health',
            'GET /players?q={name}',
            'GET /recommend?position=&style=&max_budget_eur=&max_age=&leagues=',
            'GET /similar/{player_name}',
            'GET /hidden-gems?position=&max_value_eur=&min_talent_score=&max_age=',
            'GET /compare?player_a=&player_b=',
            'GET /styles',
        ],
        'cors': 'enabled (all origins)',
        'docs': 'http://localhost:8000/docs',
    },
    'models_loaded': [
        'KNN Cosine Similarity (4,817 players)',
        'XGBoost Talent Score (R²=0.76)',
        'K-Means Style Clustering (12 styles)',
    ],
    'saved_artifacts': [
        'src/api/main.py',
        'models/recommendation_engine.pkl',
        'requirements_api.txt',
    ],
    'next': 'notebook 09 — React Frontend Dashboard',
}

print('=' * 60)
print('  NB08 COMPLETE — FASTAPI BACKEND')
print('=' * 60)
import json as _json
print(_json.dumps(summary, indent=2))
print('=' * 60)
print('  API READY — run uvicorn to start')
print('  → NEXT: notebook 09 — React Frontend')
print('=' * 60)